# Khmer Document Text-Line Detection — YOLOv11 Training on Kaggle

This notebook:
1. Installs dependencies
2. Loads the dataset from Hugging Face (either as a YOLO zip or as Parquet)
3. Trains a YOLOv11 model (configurable variant)
4. Evaluates and saves the best weights

**Before running:** set `HF_REPO_ID` in the *Configuration* cell below.

## 0 · Configuration — edit these values

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER SETTINGS  ← the only cells you need to edit
# ─────────────────────────────────────────────────────────────────────────────

# Your Hugging Face dataset repo   e.g. "myname/khmer-doc-lines"
HF_REPO_ID   = "Darayut/khmer-textline-detection"

# HF token (only needed if the repo is private).
# On Kaggle: add your token as a Secret named HF_TOKEN and leave this as None.
HF_TOKEN     = None   # or "hf_xxxx"

# YOLO model variant to train
# Options: yolo11n  yolo11s  yolo11m  yolo11l  yolo11x
MODEL        = "yolo11n.pt"

# Training hyper-parameters
EPOCHS       = 100
IMGSZ        = 800    # match your document page width
BATCH        = 8      # lower if OOM; -1 = auto
LR0          = 0.01   # initial learning rate
PATIENCE     = 20     # early stopping patience (0 = disabled)
WORKERS      = 2      # DataLoader workers (keep low on Kaggle)

# Dataset loading strategy
# "zip"    → download data/yolo_raw.zip from HF (fastest, recommended)
# "parquet" → reconstruct from HF Parquet dataset (fallback)
LOAD_MODE    = "parquet"

# Output directory for weights and results
OUTPUT_DIR   = "/kaggle/working/runs"
RUN_NAME     = "khmer_doc_lines"

## 1 · Install dependencies

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

print("Installing ultralytics …")
pip("ultralytics>=8.3.0")

print("Installing huggingface libraries …")
pip("huggingface_hub>=0.20.0", "datasets>=2.14.0")

print("Done.")

## 2 · Environment check

In [ ]:
import os, pathlib, shutil
import torch
from ultralytics import YOLO
from ultralytics import __version__ as ul_ver

print(f"PyTorch      : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Ultralytics  : {ul_ver}")

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"Training on  : {DEVICE}")

# Resolve any Kaggle secret for HF token
if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        _token = UserSecretsClient().get_secret("HF_TOKEN")
        if _token:
            HF_TOKEN = _token
            print("HF token loaded from Kaggle Secrets.")
    except Exception:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF token set.")
else:
    print("No HF token — repo must be public.")

## 3 · Download dataset from Hugging Face

In [ ]:
import io, zipfile, json, re
from pathlib import Path
from PIL import Image as PILImage

DATA_ROOT = Path("/kaggle/working/dataset")


# ── Helper: write a YOLO label file ──────────────────────────────────────────
def write_label(label_path: Path, annotations: dict) -> None:
    """Write a YOLO .txt from an annotations dict {bbox:[[cx,cy,w,h]…], cls_id:[…]}."""
    label_path.parent.mkdir(parents=True, exist_ok=True)
    bboxes  = annotations.get("bbox",   [])
    cls_ids = annotations.get("cls_id", [])
    with open(label_path, "w") as f:
        for cls_id, bbox in zip(cls_ids, bboxes):
            cx, cy, w, h = bbox
            f.write(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")


# ── Helper: write dataset.yaml ────────────────────────────────────────────────
def write_dataset_yaml(root: Path, has_val: bool) -> Path:
    yaml_path = root / "dataset.yaml"
    lines = [
        f"path: {root.resolve()}",
        "train: images/train",
    ]
    if has_val:
        lines.append("val:   images/val")
    else:
        # Point val at train so Ultralytics doesn't complain
        lines.append("val:   images/train")
    lines += ["", "nc: 1", "names:", "  0: text_line", ""]
    yaml_path.write_text("\n".join(lines))
    return yaml_path


print("Helpers defined.")

In [ ]:
# ── MODE: zip ─────────────────────────────────────────────────────────────────
# Downloads the pre-built YOLO zip from HF and extracts it directly.
# Fastest option — no image re-encoding needed.

if LOAD_MODE == "zip":
    from huggingface_hub import hf_hub_download

    print(f"Downloading yolo_raw.zip from {HF_REPO_ID} …")
    zip_path = hf_hub_download(
        repo_id   = HF_REPO_ID,
        filename  = "data/yolo_raw.zip",
        repo_type = "dataset",
        token     = HF_TOKEN,
    )
    print(f"Downloaded to: {zip_path}")

    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print("Extracting …")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_ROOT)

    # Check what splits are present
    has_val = (DATA_ROOT / "images" / "val").is_dir() and \
              any((DATA_ROOT / "images" / "val").iterdir())

    # Write / overwrite dataset.yaml with absolute paths
    yaml_path = write_dataset_yaml(DATA_ROOT, has_val)

    n_train = len(list((DATA_ROOT / "images" / "train").glob("*.jpg")))
    n_val   = len(list((DATA_ROOT / "images" / "val").glob("*.jpg"))) if has_val else 0

    print(f"\nExtracted to : {DATA_ROOT}")
    print(f"Train images : {n_train}")
    print(f"Val images   : {n_val if has_val else 'not present (using train for val)'}")
    print(f"dataset.yaml : {yaml_path}")

In [ ]:
# ── MODE: parquet ─────────────────────────────────────────────────────────────
# Loads the HF Dataset (Parquet) and reconstructs the YOLO directory layout.
# Use this if the zip was not uploaded, or if you want to stream a subset.

if LOAD_MODE == "parquet":
    from datasets import load_dataset
    import numpy as np

    print(f"Loading dataset from HF: {HF_REPO_ID} …")
    ds = load_dataset(HF_REPO_ID, token=HF_TOKEN)
    print(f"Splits available: {list(ds.keys())}")

    has_val = "val" in ds
    splits_to_write = ["train"] + (["val"] if has_val else [])

    for split in splits_to_write:
        img_dir = DATA_ROOT / "images" / split
        lbl_dir = DATA_ROOT / "labels" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        split_ds = ds[split]
        print(f"Writing {len(split_ds)} {split} examples …")

        for i, row in enumerate(split_ds):
            stem     = row["image_id"]
            pil_img  = row["image"]          # PIL Image from HF Image() feature
            anns     = row["annotations"]

            # Save image
            img_path = img_dir / f"{stem}.jpg"
            pil_img.save(str(img_path), format="JPEG", quality=93)

            # Save label
            lbl_path = lbl_dir / f"{stem}.txt"
            write_label(lbl_path, anns)

            if (i + 1) % 200 == 0 or (i + 1) == len(split_ds):
                print(f"  {split}: {i+1}/{len(split_ds)}")

    yaml_path = write_dataset_yaml(DATA_ROOT, has_val)

    n_train = len(list((DATA_ROOT / "images" / "train").glob("*.jpg")))
    n_val   = len(list((DATA_ROOT / "images" / "val").glob("*.jpg"))) if has_val else 0
    print(f"\nDataset written to: {DATA_ROOT}")
    print(f"Train: {n_train}  |  Val: {n_val if has_val else 'none (reusing train)'}")
    print(f"dataset.yaml: {yaml_path}")

In [ ]:
# Verify the dataset layout before training
import yaml

yaml_path = DATA_ROOT / "dataset.yaml"
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

print("dataset.yaml contents:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

# Quick sanity: show one sample label
sample_labels = list((DATA_ROOT / "labels" / "train").glob("*.txt"))
if sample_labels:
    sample_lbl = sample_labels[0]
    lines = sample_lbl.read_text().strip().splitlines()
    print(f"\nSample label ({sample_lbl.name}): {len(lines)} boxes")
    for ln in lines[:3]:
        print(f"  {ln}")

# Visualise one training image with its boxes
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

train_imgs = list((DATA_ROOT / "images" / "train").glob("*.jpg"))
if train_imgs:
    img_path = train_imgs[0]
    lbl_path = DATA_ROOT / "labels" / "train" / (img_path.stem + ".txt")
    img = PILImage.open(img_path)
    iw, ih = img.size

    fig, ax = plt.subplots(1, 1, figsize=(10, 14))
    ax.imshow(img, cmap="gray" if img.mode == "L" else None)

    if lbl_path.exists():
        for ln in lbl_path.read_text().strip().splitlines():
            parts = ln.split()
            if len(parts) != 5:
                continue
            _, cx, cy, w, h = map(float, parts)
            x1 = (cx - w / 2) * iw
            y1 = (cy - h / 2) * ih
            rect = mpatches.Rectangle(
                (x1, y1), w * iw, h * ih,
                linewidth=1.5, edgecolor="lime", facecolor="none"
            )
            ax.add_patch(rect)

    ax.set_title(f"{img_path.name}  ({iw}×{ih})")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/sample_annotation.png", dpi=120)
    plt.show()
    print("Sample saved to /kaggle/working/sample_annotation.png")

## 4 · Train YOLOv11

In [ ]:
from ultralytics import YOLO, settings as ul_settings

# Disable integrations we don't need on Kaggle
ul_settings.update({
    "mlflow":      False,
    "wandb":       False,
    "tensorboard": False,
    "clearml":     False,
    "comet":       False,
})

yaml_path = DATA_ROOT / "dataset.yaml"

print(f"Model        : {MODEL}")
print(f"Epochs       : {EPOCHS}")
print(f"Image size   : {IMGSZ}")
print(f"Batch        : {BATCH}")
print(f"Device       : {DEVICE}")
print(f"Output dir   : {OUTPUT_DIR}/{RUN_NAME}")
print(f"Dataset yaml : {yaml_path}")
print()

model = YOLO(MODEL)

results = model.train(
    data        = str(yaml_path),
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH,
    device      = DEVICE,
    lr0         = LR0,
    patience    = PATIENCE,
    workers     = WORKERS,
    project     = OUTPUT_DIR,
    name        = RUN_NAME,
    exist_ok    = True,
    verbose     = True,
    # Augmentation — photometric only (spatial distortion would break line bbox)
    degrees     = 0.0,    # no rotation
    shear       = 0.0,    # no shear
    perspective = 0.0,    # no perspective
    fliplr      = 0.0,    # no horizontal flip (documents have fixed orientation)
    flipud      = 0.0,    # no vertical flip
    mosaic      = 0.5,    # mosaic helps with scale variety
    mixup       = 0.05,   # mild mixup
    hsv_h       = 0.01,
    hsv_s       = 0.3,
    hsv_v       = 0.3,
)

print("\nTraining complete.")

## 5 · Evaluate and inspect results

In [ ]:
import pandas as pd

run_dir   = Path(OUTPUT_DIR) / RUN_NAME
best_pt   = run_dir / "weights" / "best.pt"
res_csv   = run_dir / "results.csv"

print(f"Run dir  : {run_dir}")
print(f"Best weights: {best_pt}  (exists={best_pt.exists()})")

# ── Training curves ───────────────────────────────────────────────────────────
if res_csv.exists():
    df = pd.read_csv(res_csv)
    df.columns = [c.strip() for c in df.columns]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    pairs = [
        ("train/box_loss",          "Train box loss"),
        ("train/cls_loss",          "Train cls loss"),
        ("train/dfl_loss",          "Train dfl loss"),
        ("metrics/mAP50(B)",        "Val mAP50"),
        ("metrics/mAP50-95(B)",     "Val mAP50-95"),
        ("metrics/precision(B)",    "Val Precision"),
    ]
    for ax, (col, title) in zip(axes.flatten(), pairs):
        if col in df.columns:
            ax.plot(df["epoch"], df[col])
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)

    plt.suptitle(f"{MODEL} — Training Curves", fontsize=13)
    plt.tight_layout()
    curve_path = "/kaggle/working/training_curves.png"
    plt.savefig(curve_path, dpi=120)
    plt.show()
    print(f"Curves saved: {curve_path}")

    # Final metrics
    last = df.iloc[-1]
    print(f"\nFinal epoch metrics:")
    for col in ["metrics/mAP50(B)","metrics/mAP50-95(B)",
                "metrics/precision(B)","metrics/recall(B)"]:
        if col in df.columns:
            print(f"  {col:<30} {last[col]:.4f}")

In [ ]:
# ── Run val with best weights ─────────────────────────────────────────────────
if best_pt.exists():
    best_model = YOLO(str(best_pt))
    val_result = best_model.val(
        data    = str(DATA_ROOT / "dataset.yaml"),
        imgsz   = IMGSZ,
        device  = DEVICE,
        verbose = True,
    )
    print(f"\nmAP50    : {val_result.box.map50:.4f}")
    print(f"mAP50-95 : {val_result.box.map:.4f}")
else:
    print("best.pt not found — training may not have completed.")

In [ ]:
# ── Visual inference on a few val / train images ─────────────────────────────
if best_pt.exists():
    best_model = YOLO(str(best_pt))

    # Grab up to 4 sample images
    sample_dir = DATA_ROOT / "images" / ("val" if has_val else "train")
    samples    = sorted(sample_dir.glob("*.jpg"))[:4]

    fig, axes = plt.subplots(1, len(samples), figsize=(6 * len(samples), 9))
    if len(samples) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, samples):
        res = best_model.predict(
            str(img_path), conf=0.30, iou=0.45, verbose=False
        )[0]
        annotated = res.plot(line_width=2, font_size=8)
        ax.imshow(annotated[:, :, ::-1])   # BGR → RGB
        ax.set_title(f"{img_path.name}\n{len(res.boxes)} boxes", fontsize=9)
        ax.axis("off")

    plt.suptitle("Inference samples (best.pt)", fontsize=12)
    plt.tight_layout()
    pred_path = "/kaggle/working/predictions.png"
    plt.savefig(pred_path, dpi=120)
    plt.show()
    print(f"Predictions saved: {pred_path}")

## 6 · Save artefacts for download

In [ ]:
import shutil

out = Path("/kaggle/working")

# Copy best and last weights to /kaggle/working for easy download
for name in ["best.pt", "last.pt"]:
    src = run_dir / "weights" / name
    if src.exists():
        shutil.copy(src, out / name)
        print(f"Copied: {name}")

# Copy confusion matrix and PR curve if present
for png in run_dir.glob("*.png"):
    shutil.copy(png, out / png.name)

# List all output files
print("\nFiles in /kaggle/working:")
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:<40} {f.stat().st_size / 1e6:.2f} MB")

## 7 · (Optional) Push best weights back to Hugging Face

Uncomment the cell below to push `best.pt` to a model repo on the Hub.

In [ ]:
# UNCOMMENT to push best.pt back to HuggingFace
#
# MODEL_REPO_ID = "YOUR_USERNAME/khmer-doc-lines-yolo11n"
#
# from huggingface_hub import HfApi
# api = HfApi(token=HF_TOKEN)
# api.create_repo(MODEL_REPO_ID, repo_type="model", exist_ok=True)
# api.upload_file(
#     path_or_fileobj = str(out / "best.pt"),
#     path_in_repo    = "best.pt",
#     repo_id         = MODEL_REPO_ID,
#     repo_type       = "model",
# )
# print(f"Pushed to: https://huggingface.co/{MODEL_REPO_ID}")